In [ ]:
import random
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
from IPython.display import clear_output


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


class SnakeEnv:
    def __init__(self, size=10, device="cpu"):
        self.size = size
        self.device = device
        self.max_steps = size * size * 4
        self.reset()

    def reset(self):
        self.snake = [(5, 5), (5, 4), (5, 3)]
        self.direction = (0, 1)
        self.done = False
        self.steps = 0
        self.spawn_food()
        return self.get_state()

    def spawn_food(self):
        while True:
            self.food = (random.randint(0, self.size - 1), random.randint(0, self.size - 1))
            if self.food not in self.snake:
                break

    def step(self, action):
        """
        action:
            0 = Straight
            1 = Right
            2 = Left
        """
        clock_wise = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        idx = clock_wise.index(self.direction)

        if action == 0:
            new_dir = clock_wise[idx]
        elif action == 1:
            new_dir = clock_wise[(idx + 1) % 4]
        else:
            new_dir = clock_wise[(idx - 1) % 4]

        old_dist = abs(self.snake[0][0] - self.food[0]) + abs(self.snake[0][1] - self.food[1])

        self.direction = new_dir
        head = (self.snake[0][0] + self.direction[0],
                self.snake[0][1] + self.direction[1])

        self.steps += 1

        # collision or out of bounds
        if (
            head in self.snake
            or head[0] < 0 or head[0] >= self.size
            or head[1] < 0 or head[1] >= self.size
        ):
            self.done = True
            return self.get_state(), torch.tensor(-1.0, device=self.device), True

        # time limit
        if self.steps >= self.max_steps:
            self.done = True
            return self.get_state(), torch.tensor(-0.5, device=self.device), True

        self.snake.insert(0, head)

        # small step penalty
        reward = -0.01

        if head == self.food:
            reward = 1.0
            self.spawn_food()
        else:
            self.snake.pop()
            new_dist = abs(head[0] - self.food[0]) + abs(head[1] - self.food[1])

            # small shaping
            if new_dist < old_dist:
                reward += 0.02
            else:
                reward -= 0.02

        return self.get_state(), torch.tensor(reward, device=self.device, dtype=torch.float32), False

    def get_state(self):
        head = self.snake[0]

        dir_l = self.direction == (0, -1)
        dir_r = self.direction == (0, 1)
        dir_u = self.direction == (-1, 0)
        dir_d = self.direction == (1, 0)

        def is_unsafe(p):
            return (
                p[0] < 0 or p[0] >= self.size or
                p[1] < 0 or p[1] >= self.size or
                p in self.snake
            )

        p_straight = (head[0] + self.direction[0], head[1] + self.direction[1])

        clock_wise = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        idx = clock_wise.index(self.direction)
        p_right = (head[0] + clock_wise[(idx + 1) % 4][0], head[1] + clock_wise[(idx + 1) % 4][1])
        p_left = (head[0] + clock_wise[(idx - 1) % 4][0], head[1] + clock_wise[(idx - 1) % 4][1])

        state = [
            float(is_unsafe(p_straight)),
            float(is_unsafe(p_right)),
            float(is_unsafe(p_left)),
            float(dir_l), float(dir_r), float(dir_u), float(dir_d),
            float(self.food[0] < head[0]),
            float(self.food[0] > head[0]),
            float(self.food[1] < head[1]),
            float(self.food[1] > head[1]),
        ]
        return torch.tensor(state, dtype=torch.float32, device=self.device)


class Actor(nn.Module):
    def __init__(self, state_dim=11, action_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim),
        )

    def forward(self, x):
        return self.net(x)  # logits


class Critic(nn.Module):
    def __init__(self, state_dim=11):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def compute_gae(rewards, dones, values, next_value, gamma=0.99, lmbda=0.95):
    """
    rewards: Tensor [T]
    dones:   Tensor [T] with 0.0 or 1.0
    values:  Tensor [T]
    next_value: scalar tensor or float
    """
    next_value = torch.as_tensor(next_value, device=rewards.device, dtype=rewards.dtype)
    values = torch.cat([values, next_value.unsqueeze(0)], dim=0)

    advantages = torch.zeros_like(rewards)
    gae = 0.0

    for t in reversed(range(len(rewards))):
        delta = rewards[t] + gamma * values[t + 1] * (1.0 - dones[t]) - values[t]
        gae = delta + gamma * lmbda * (1.0 - dones[t]) * gae
        advantages[t] = gae

    returns = advantages + values[:-1]
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    return returns, advantages

def select_action(actor, state):
    """
    state: Tensor [11]
    returns: action(int), log_prob(scalar tensor)
    """
    with torch.no_grad():
        logits = actor(state.unsqueeze(0))          # [1, 3]
        dist = Categorical(logits=logits)
        action = dist.sample().squeeze(0)           # scalar tensor
        log_prob = dist.log_prob(action.unsqueeze(0)).squeeze(0)
    return action.item(), log_prob


def render_game(env):
    board = [['.' for _ in range(env.size)] for _ in range(env.size)]

    fr, fc = env.food
    board[fr][fc] = 'F'

    for i, (r, c) in enumerate(env.snake):
        board[r][c] = 'H' if i == 0 else 'S'

    print(f"Score: {len(env.snake) - 3}")
    for row in board:
        print(" ".join(row))
    print()


env = SnakeEnv(size=10, device=device)

actor = Actor().to(device)
critic = Critic().to(device)

actor_optimizer = optim.Adam(actor.parameters(), lr=3e-4)
critic_optimizer = optim.Adam(critic.parameters(), lr=1e-4)

num_updates = 3000
rollout_steps = 2048
mini_batch_size = 256
update_epochs = 10

eps_clip = 0.2
gamma = 0.99
lmbda = 0.95
entropy_coef = 0.01
max_grad_norm = 0.5

best_mean_return = -1e9

for update in range(1, num_updates + 1):
    states = []
    actions = []
    rewards = []
    dones = []
    old_log_probs = []

    state = env.reset()

    episode_returns = []
    episode_return = 0.0
    episode_lengths = []
    episode_len = 0

    # Collect rollout
    for _ in range(rollout_steps):
        action, old_log_prob = select_action(actor, state)

        next_state, reward, done = env.step(action)

        states.append(state)
        actions.append(torch.tensor(action, device=device, dtype=torch.long))
        rewards.append(reward)
        dones.append(torch.tensor(float(done), device=device))
        old_log_probs.append(old_log_prob.detach())

        episode_return += reward.item()
        episode_len += 1

        state = next_state

        if done:
            episode_returns.append(episode_return)
            episode_lengths.append(episode_len)
            episode_return = 0.0
            episode_len = 0
            state = env.reset()

    # Stack rollout
    states_t = torch.stack(states)                # [T, 11]
    actions_t = torch.stack(actions)              # [T]
    rewards_t = torch.stack(rewards)              # [T]
    dones_t = torch.stack(dones)                  # [T]
    old_log_probs_t = torch.stack(old_log_probs)  # [T]

    with torch.no_grad():
        values = critic(states_t)                 # [T]
        next_value = critic(state.unsqueeze(0)).item()
        returns, advantages = compute_gae(
            rewards_t, dones_t, values, next_value, gamma=gamma, lmbda=lmbda
        )

    # PPO update
    total_actor_loss = 0.0
    total_critic_loss = 0.0
    total_entropy = 0.0
    update_steps = 0

    for _ in range(update_epochs):
        indices = torch.randperm(rollout_steps, device=device)

        for start in range(0, rollout_steps, mini_batch_size):
            mb_idx = indices[start:start + mini_batch_size]

            mb_states = states_t[mb_idx]
            mb_actions = actions_t[mb_idx]
            mb_old_log_probs = old_log_probs_t[mb_idx]
            mb_returns = returns[mb_idx]
            mb_advantages = advantages[mb_idx]

            # Actor update
            logits = actor(mb_states)
            dist = Categorical(logits=logits)
            new_log_probs = dist.log_prob(mb_actions)
            entropy = dist.entropy().mean()

            ratio = torch.exp(new_log_probs - mb_old_log_probs)
            surr1 = ratio * mb_advantages
            surr2 = torch.clamp(ratio, 1 - eps_clip, 1 + eps_clip) * mb_advantages

            actor_loss = -torch.min(surr1, surr2).mean() - entropy_coef * entropy

            actor_optimizer.zero_grad()
            actor_loss.backward()
            torch.nn.utils.clip_grad_norm_(actor.parameters(), max_grad_norm)
            actor_optimizer.step()

            # Critic update
            values_pred = critic(mb_states)
            critic_loss = 0.5 * (mb_returns - values_pred).pow(2).mean()

            critic_optimizer.zero_grad()
            critic_loss.backward()
            torch.nn.utils.clip_grad_norm_(critic.parameters(), max_grad_norm)
            critic_optimizer.step()

            total_actor_loss += actor_loss.item()
            total_critic_loss += critic_loss.item()
            total_entropy += entropy.item()
            update_steps += 1

    mean_ep_return = float(np.mean(episode_returns)) if len(episode_returns) > 0 else 0.0
    max_ep_return = float(np.max(episode_returns)) if len(episode_returns) > 0 else 0.0
    mean_ep_len = float(np.mean(episode_lengths)) if len(episode_lengths) > 0 else 0.0

    if mean_ep_return > best_mean_return:
        best_mean_return = mean_ep_return
        torch.save({
            "actor": actor.state_dict(),
            "critic": critic.state_dict(),
        }, "snake_ppo_best.pt")

    if update % 10 == 0:
        print(
            f"Update {update:4d} | "
            f"MeanEpReturn: {mean_ep_return:.3f} | "
            f"MaxEpReturn: {max_ep_return:.3f} | "
            f"MeanEpLen: {mean_ep_len:.1f} | "
            f"ActorLoss: {total_actor_loss / update_steps:.4f} | "
            f"CriticLoss: {total_critic_loss / update_steps:.4f} | "
            f"Entropy: {total_entropy / update_steps:.4f} | "
            f"EntropyCoef: {entropy_coef:.4f}"
        )

    # mild entropy decay
    entropy_coef = max(0.003, entropy_coef * 0.9995)

Using device: cuda
Update   10 | MeanEpReturn: 7.789 | MaxEpReturn: 17.890 | MeanEpLen: 75.1 | ActorLoss: -0.0204 | CriticLoss: 0.8970 | Entropy: 0.5260 | EntropyCoef: 0.0100
Update   20 | MeanEpReturn: 13.489 | MaxEpReturn: 22.810 | MeanEpLen: 109.7 | ActorLoss: -0.0104 | CriticLoss: 1.6635 | Entropy: 0.3726 | EntropyCoef: 0.0099
Update   30 | MeanEpReturn: 16.274 | MaxEpReturn: 27.490 | MeanEpLen: 137.2 | ActorLoss: -0.0111 | CriticLoss: 1.8015 | Entropy: 0.3464 | EntropyCoef: 0.0099
Update   40 | MeanEpReturn: 14.297 | MaxEpReturn: 23.970 | MeanEpLen: 129.2 | ActorLoss: -0.0127 | CriticLoss: 1.7631 | Entropy: 0.3401 | EntropyCoef: 0.0098
Update   50 | MeanEpReturn: 18.356 | MaxEpReturn: 30.390 | MeanEpLen: 160.5 | ActorLoss: -0.0071 | CriticLoss: 1.5617 | Entropy: 0.2993 | EntropyCoef: 0.0098
Update   60 | MeanEpReturn: 17.631 | MaxEpReturn: 24.320 | MeanEpLen: 162.2 | ActorLoss: -0.0084 | CriticLoss: 1.5399 | Entropy: 0.2940 | EntropyCoef: 0.0097
Update   70 | MeanEpReturn: 18.522 

In [5]:

print("\nTraining done. Starting demo...")

# load best checkpoint if exists
try:
    ckpt = torch.load("snake_ppo_best.pt", map_location=device)
    actor.load_state_dict(ckpt["actor"])
    critic.load_state_dict(ckpt["critic"])
    print("Loaded best checkpoint.")
except Exception:
    print("No checkpoint loaded, using current weights.")

s = env.reset()
done = False

try:
    while not done:
        clear_output(wait=True)
        render_game(env)

        with torch.no_grad():
            logits = actor(s.unsqueeze(0))
            a = torch.argmax(logits, dim=-1).item()

        s, r, done = env.step(a)
        time.sleep(0.15)

    clear_output(wait=True)
    render_game(env)
    print("Game Over!")
except KeyboardInterrupt:
    print("Stop game!")

Score: 23
. . . . . . . . . .
. . S S . . . . . .
. . S S . . . . . .
. . S S . . . . . .
. . S S . . . . . .
. S S S S S S . . .
. S S S S H S . . .
. S S S S S S . . .
F . . . . . . . . .
. . . . . . . . . .

Game Over!
